In [2]:
import pandas as pd
from pathlib import Path

# Measurements

In [110]:
old_data = Path('data')
measurement_folder_old = old_data / 'irradiation_measurements'
mak_measurement_old = measurement_folder_old / 'MAK_physics_dept'
ministry_energy_measurement_old = measurement_folder_old / 'ministry_energy_ug'
CBE_measurement_old = measurement_folder_old / 'CBE_Data'

new_data = Path('Data')
measurement_folder_new = new_data / 'irradiation_measurements'
mak_measurement_new = measurement_folder_new / 'MAK_physics_dept'
ministry_energy_measurement_new = measurement_folder_new / 'ministry_energy_ug'
CBE_measurement_new = measurement_folder_new / 'CBE_Data'

In [174]:
def read_and_normalize_csvs(filepaths, country='Unidentified'):
    dataframes = []
    
    for file_path in [filepaths] if not isinstance(filepaths, list) else filepaths:
        df = pd.read_csv(file_path, index_col=False)
        
        # Extract location
        if 'location' in df.columns:
            location = country +  '_' + df['location'][0].lower()
        else:
            location = country
            
        # Identify datetime column
        if 'Day' in df.columns:
            df['datetime'] = pd.to_datetime(df['Day'], errors="coerce")
        elif 'MEASURE_DATE' in df.columns:
            df['datetime'] = pd.to_datetime(df['MEASURE_DATE'], errors='coerce')
        elif 'datetime' in df.columns:
            df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
        else:
            print(f"Warning: No recognized datetime column in {file_path.name}")
            continue

        # Extract GHI column
        if 'GHI_1 (kWh/m2/day)' in df.columns:
            df = df[['datetime', 'GHI_1 (kWh/m2/day)']]
            df.rename(columns={'GHI_1 (kWh/m2/day)': 'ghi'}, inplace=True)
            df['ghi'] = df['ghi'] * 1000
        elif 'MEASURE_VALUE' in df.columns:
            df = df[['datetime', 'MEASURE_VALUE']]
            df.rename(columns={'MEASURE_VALUE': 'ghi'}, inplace=True)
        elif 'GHI (W/m2)' in df.columns:
            df.index = df['datetime']
            df['ghi'] = df['GHI (W/m2)'].resample('D').sum()
            df.dropna(inplace=True)
            df.sort_index(inplace=True)
            df = df[['datetime', 'ghi']]
        else:
            print(f"Warning: No GHI column found in {file_path.name}")
            continue
        
        df['location'] = location
        df.dropna(inplace=True)
        df = df[df['ghi'] > 0]
        dataframes.append(df)
        
    return pd.concat(dataframes, ignore_index=True) if dataframes else None

## Ministry 

In [111]:
soroti_measurements_old = ministry_energy_measurement_old / 'Soroti'
wadelai_measurements_old = ministry_energy_measurement_old / 'Wadelai'

In [93]:
soroti_filepaths = list(soroti_measurements_old.glob('*.csv'))
wadelai_filepaths = list(wadelai_measurements_old.glob('*.csv'))

In [113]:
soroti = read_and_normalize_csvs(soroti_filepaths, 'soroti')
soroti.to_csv(ministry_energy_measurement_new / 'soroti.csv', index=False)

In [112]:
wadelai = read_and_normalize_csvs(wadelai_filepaths, 'wadelai')
wadelai.to_csv(ministry_energy_measurement_new / 'wadelai.csv', index=False)

## CBE

In [114]:
egypt_measurements = CBE_measurement_old / 'Egypt'
ghana_measurements = CBE_measurement_old / 'Ghana'
kenya_measurements = CBE_measurement_old / 'Kenya'
madagascar_measurements = CBE_measurement_old / 'Madagascar'
nigeria_measurements = CBE_measurement_old / 'Nigeria'
somalia_measurements = CBE_measurement_old / 'Somalia'

In [90]:
egypt_filepaths = list(egypt_measurements.glob('*.csv'))
ghana_filepaths = list(ghana_measurements.glob('*.csv'))
kenya_filepaths = list(kenya_measurements.glob('*.csv'))
madagascar_filepaths = list(madagascar_measurements.glob('*.csv'))
nigeria_filepaths = list(nigeria_measurements.glob('*.csv'))
somalia_filepaths = list(somalia_measurements.glob('*.csv'))

In [115]:
egypt = read_and_normalize_csvs(egypt_filepaths, 'egypt')
ghana = read_and_normalize_csvs(ghana_filepaths, 'ghana')
kenya = read_and_normalize_csvs(kenya_filepaths, 'kenya')
madagascar = read_and_normalize_csvs(madagascar_filepaths, 'madagascar')
nigeria = read_and_normalize_csvs(nigeria_filepaths, 'nigeria')
somalia = read_and_normalize_csvs(somalia_filepaths, 'somalia')

egypt.to_csv(CBE_measurement_new / 'egypt.csv', index=False)
ghana.to_csv(CBE_measurement_new / 'ghana.csv', index=False)
kenya.to_csv(CBE_measurement_new / 'kenya.csv', index=False)
nigeria.to_csv(CBE_measurement_new / 'nigeria.csv', index=False)
somalia.to_csv(CBE_measurement_new / 'somalia.csv', index=False)
madagascar.to_csv(CBE_measurement_new / 'madagascar.csv', index=False)

## Mak

In [124]:
kampala_path = mak_measurement_old / "kampala_data.csv"
lira_path = mak_measurement_old / "lira_data.csv"
mbarara_path = mak_measurement_old / "mbarara_data.csv"
tororo_path = mak_measurement_old / "tororo_data.csv"


In [185]:
kampala = read_and_normalize_csvs(mbarara_path)
kampala.tail()

,datetime,ghi,location
1491,2016-04-19,88.8693,Unidentified
1492,2016-04-20,17692.4132,Unidentified
1493,2016-04-21,35997.7100,Unidentified
1494,2016-04-22,70402.5850,Unidentified
1495,2016-04-23,4472.4340,Unidentified


In [183]:
kampala = read_and_normalize_csvs(kampala_path, 'kampala')
lira = read_and_normalize_csvs(lira_path, 'lira')
tororo = read_and_normalize_csvs(tororo_path, 'tororo')

lira.to_csv(mak_measurement_new / 'lira.csv', index=False)
tororo.to_csv(mak_measurement_new / 'tororo.csv', index=False)
kampala.to_csv(mak_measurement_new / 'kampala.csv', index=False)

# Estimates

In [190]:
old_data = Path('data')
estimates_folder_old = old_data / 'irradiation_estimates'
mak_estimates_old = estimates_folder_old / 'MAK_physics_dept'
ministry_energy_estimates_old = estimates_folder_old / 'ministry_energy_ug'
CBE_estimates_old = estimates_folder_old / 'CBE_Data'

new_data = Path('Data')
estimates_folder_new = new_data / 'irradiation_estimates'
mak_estimates_new = estimates_folder_new / 'MAK_physics_dept'
ministry_energy_estimates_new = estimates_folder_new / 'ministry_energy_ug'
CBE_estimates_new = estimates_folder_new / 'CBE_Data'

## Ministry

In [191]:
soroti_estimates_old = ministry_energy_estimates_old / 'Soroti'
wadelai_estimates_old = ministry_energy_estimates_old / 'Wadelai'

In [193]:
soroti_filepaths = list(soroti_estimates_old.glob('*.csv'))
wadelai_filepaths = list(wadelai_estimates_old.glob('*csv'))
wadelai_filepaths

[PosixPath('data/irradiation_estimates/ministry_energy_ug/Wadelai/Wadelai_CAMS-RAD2020.csv'),
 PosixPath('data/irradiation_estimates/ministry_energy_ug/Wadelai/Wadelai_CAMS-RAD2021.csv'),
 PosixPath('data/irradiation_estimates/ministry_energy_ug/Wadelai/Wadelai_NREL2020.csv'),
 PosixPath('data/irradiation_estimates/ministry_energy_ug/Wadelai/Wadelai_NREL2021.csv'),
 PosixPath('data/irradiation_estimates/ministry_energy_ug/Wadelai/Wadelai_solcast2020.csv'),
 PosixPath('data/irradiation_estimates/ministry_energy_ug/Wadelai/Wadelai_solcast2021.csv')]